# 02a — Vectorized phenotype extraction

## Purpose
Extract configured clinical phenotypes from problems, diagnoses, and prescribing data, evaluate their temporal windows, aggregate composite phenotypes, and save reusable parquet artifacts.

In [46]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/app/notebooks/data/synthetic/10000p")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using data from: {DATA_DIR}")

from icare_risk.utils.diagnostics import (
    check_df,
    check_value_counts,
    check_merge_keys,
)
from icare_risk.utils.io import load_pkg_yaml
from icare_risk.clinphen.utils.vectorized import (
    run_vectorized,
    evaluate_temporal_windows,
    aggregate_composite_phenotypes,
    parse_phenotypes_to_batches
)

Using data from: /app/notebooks/data/synthetic/10000p


## Load source tables

In [42]:
df_episodes = pd.read_csv(
    DATA_DIR / "icare_episodes_anon.csv",
    parse_dates=["ADMISSION_DATE"],
)
df_problems = pd.read_csv(
    DATA_DIR / "icare_problems_anon.csv",
    parse_dates=["PROBLEM_DT_TM"],
)
df_diagnoses = pd.read_csv(
    DATA_DIR / "icare_episodes_diagnosis_anon.csv",
    parse_dates=["DIAGNOSIS_DATE"],
)
df_prescribing = pd.read_csv(
    DATA_DIR / "icare_pharmacy_prescribing_anon.csv",
    parse_dates=["ORDER_DT_TM"],
)

print(f"Episodes:    {df_episodes.shape}")
print(f"Problems:    {df_problems.shape}")
print(f"Diagnoses:   {df_diagnoses.shape}")
print(f"Prescribing: {df_prescribing.shape}")

check_df(
    df_episodes,
    "Episodes input",
    columns=[
        "SUBJECT",
        "SPELL_IDENTIFIER",
        "ADMISSION_DATE",
        "DISCHARGE_DATE",
    ],
    n=5,
)

Episodes:    (10000, 11)
Problems:    (30208, 6)
Diagnoses:   (30051, 12)
Prescribing: (69533, 16)

CHECK: Episodes input
Shape: 10,000 rows x 11 columns

Preview:


,SUBJECT,SPELL_IDENTIFIER,ADMISSION_DATE,DISCHARGE_DATE
0,10001,7929089,2020-06-10,2020-07-03
1,10002,8200425,2020-03-27,2020-03-28
2,10003,9931252,2023-01-06,2023-01-25
3,10004,5095152,2022-09-14,2022-10-12
4,10005,7807609,2023-09-08,2023-09-23


## Configure domains and phenotypes

In [48]:
domain_registry = {
    "prescribing": {
        "df": df_prescribing,
        "subject_col": "SUBJECT",
        "time_col": "ORDER_DT_TM",
        "target_col": "MEDICATION_NAME",
        "match_type": "contains",
    },
    "problems": {
        "df": df_problems,
        "subject_col": "SUBJECT",
        "time_col": "PROBLEM_DT_TM",
        "target_col": "PROBLEM_CODE",
    },
    "diagnosis": {
        "df": df_diagnoses,
        "subject_col": "SUBJECT",
        "time_col": "DIAGNOSIS_DATE",
        "sources": [
            {"target_col": "DIAGNOSIS_CODE_SNOMED", "match_type": "exact"},
            {"target_col": "DIAGNOSIS_CODE_ICD",    "match_type": "prefix"},
        ],
    },
}

all_phenotypes = load_pkg_yaml("config/icare/phenotypes.yaml")
charlson = {k:v for k,v in all_phenotypes.items() if k.startswith('charlson_')}
phenotypes = all_phenotypes  # Replace with a filtered dictionary for a smaller run.

print(f"Configured phenotypes: {len(phenotypes):,}")

Configured phenotypes: 66


## Extract and window events

In [49]:
# 1. Extract raw historical events
df_events = run_vectorized(
    domains=domain_registry,
    phenotypes=phenotypes,
    verbose=1,
)

# 2. Create index_admission column
df_episodes["index_admission"] = pd.to_datetime(
    df_episodes["ADMISSION_DATE"].astype(str) + " "
    + df_episodes["ADMISSION_TIME"].astype(str),
    errors="coerce",
)

# 3. Build the window configuration mapping from your phenotypes YAML dictionary
window_configs = {}
for batch in parse_phenotypes_to_batches(phenotypes):
    for sub_event_name, config in batch["configs"].items():
        if config.get("window") is not None:
            window_configs[sub_event_name] = config["window"]

# 4. Filter events given the episode and the windows
df_events_windowed = evaluate_temporal_windows(
    episodes_df=df_episodes,
    events_df=df_events,
    window_configs=window_configs,
    subject_col="SUBJECT",
    admission_col="index_admission",
    event_col="event_name",
    event_time_col="event_dt_tm",
    prefix="hx_",
)

# 5. Aggregate component-level binary indications into parent clinical phenotypes
df_aggregated = aggregate_composite_phenotypes(
    df=df_events_windowed,
    phenotype_configs=phenotypes,
    prefix="",
    drop_components=True,
)

print(f"Raw events:       {df_events.shape}")
print(f"Windowed events:  {df_events_windowed.shape}")
print(f"Aggregated output:{df_aggregated.shape}")


--- Running Vectorized Bench ---

--- 1. Parsing Phenotypes YAML ---
Generated 7 execution batch(es).

--- 2. Executing Batches ---

--- DEBUG BATCH 1 (Source 1) [problems -> PROBLEM_CODE] ---
Match Type: exact, Case Sensitive: False
YAML is looking for (first 5):
  - charlson_hx_mi__comp0
  - charlson_hx_mi__comp1
  - charlson_hx_chf__comp0
  - charlson_hx_chf__comp1
  - charlson_hx_pvd__comp0
DataFrame top 5 most common values (with counts):
  - 77386006: 4,531 occurrence(s)
  - none: 1,489 occurrence(s)
  - 38341003: 1,133 occurrence(s)
  - 38907003: 687 occurrence(s)
  - 195967001: 575 occurrence(s)
-> Matched 2217 record(s) using column 'PROBLEM_CODE' with match_type='exact'.

--- DEBUG BATCH 2 (Source 1) [prescribing -> MEDICATION_NAME] ---
Match Type: contains, Case Sensitive: False
YAML is looking for (first 5):
  - charlson_hx_chf__comp2
  - charlson_hx_diabetes_uncomp__comp2
  - gavaghan_hx_prior_antibiotics_fq_ceph_90d__comp0
  - pitt_hypo_score_rule__comp1
DataFrame top 5 

In [53]:
# Local Libraries
from icare_risk.clinphen.scores.gavaghan import compute_gavaghan_score
from icare_risk.clinphen.scores.holmgren import compute_holmgren_score
from icare_risk.clinphen.scores.charlson import compute_charlson_score
from icare_risk.clinphen.scores.jones import compute_jones_score
from icare_risk.clinphen.scores.tumbarello import compute_tumbarello_score
from icare_risk.clinphen.scores.kim import compute_kim_score

df_aggregated['charlson_score'] = compute_charlson_score(df_aggregated, verbose=0)
df_aggregated['gavaghan_score'] = compute_gavaghan_score(df_aggregated, verbose=0)
df_aggregated['holmgren_score'] = compute_holmgren_score(df_aggregated, verbose=0)
df_aggregated['jones_score'] = compute_jones_score(df_aggregated, verbose=0)
df_aggregated['tumbarello_score'] = compute_tumbarello_score(df_aggregated, verbose=0)
df_aggregated['kim_score'] = compute_kim_score(df_aggregated, verbose=0)

## Save artifacts

In [54]:
from icare_risk.clinphen.utils.io import save_phenotype_artifacts

save_phenotype_artifacts(
    df_raw=df_events,
    df_windowed=df_events_windowed,
    df_aggregated=df_aggregated,
    output_dir=DATA_DIR,
    prefix="02a_derived_",
)


--- Saving Phenotype Artifacts ---
 Saved Raw extracted matching events (142,810 rows)
   └─ Path: /app/notebooks/data/synthetic/10000p/02a_derived_events_extracted_raw.parquet
 Saved Window-filtered events (10,000 rows)
   └─ Path: /app/notebooks/data/synthetic/10000p/02a_derived_events_extracted_windowed.parquet
 Saved Aggregated patient/spell phenotypes (10,000 rows)
   └─ Path: /app/notebooks/data/synthetic/10000p/02a_derived_phenotypes_aggregated.parquet
-----------------------------------



## Visually debugging with total counts

In [51]:
def event_summary_v1(df_events):
    return (
        df_events.groupby("event_name")
        .agg(
            total_records=("SUBJECT", "count"),
            unique_patients=("SUBJECT", "nunique"),
        )
        .reset_index()
        .sort_values("unique_patients", ascending=False)
    ).to_string(index=False)

def event_summary_v2(df_events_windowed):
    # 1. Identify all columns starting with 'hx_'
    hx_cols = [col for col in df_events_windowed.columns if col.startswith("hx_")]

    # 2. Group by SUBJECT, check if any row per patient is True/1, then sum
    return (
        df_events_windowed[["SUBJECT"] + hx_cols]
        .groupby("SUBJECT")
        .any()  # True if the patient has a 1 in at least one episode
        .sum()  # Counts total unique patients with True per column
        .reset_index(name="patient_count")
        .rename(columns={"index": "phenotype"})
        .sort_values("patient_count", ascending=False)
    ).to_string(index=False)

In [52]:
print(event_summary_v1(df_events))

                                      event_name  total_records  unique_patients
              charlson_hx_diabetes_uncomp__comp2          19884             8468
             kim_hx_prior_antibiotics_90d__comp0          20102             8419
            kim_hx_recent_antibiotics_90d__comp1          20102             8419
gavaghan_hx_prior_antibiotics_fq_ceph_90d__comp1          20102             8419
             tumbarello_hx_recent_abx_90d__comp0          20102             8419
           jones_hx_prior_antibiotics_30d__comp1          20102             8419
            kim_hx_recent_antibiotics_90d__comp0          10098             6238
           jones_hx_prior_antibiotics_30d__comp0          10098             6238
              charlson_hx_diabetes_uncomp__comp0            828              803
                charlson_hx_renal_mod_sev__comp0            240              236
                    charlson_hx_pulmonary__comp0            175              174
                     charlso

In [8]:
print(event_summary_v2(df_events_windowed))

                                          phenotype  patient_count
             hx_kim_hx_prior_antibiotics_90d__comp0           5550
           hx_jones_hx_prior_antibiotics_30d__comp1           5550
            hx_kim_hx_recent_antibiotics_90d__comp1           5550
hx_gavaghan_hx_prior_antibiotics_fq_ceph_90d__comp1           5550
             hx_tumbarello_hx_recent_abx_90d__comp0           5550
              hx_charlson_hx_diabetes_uncomp__comp2           5531
            hx_kim_hx_recent_antibiotics_90d__comp0           3658
           hx_jones_hx_prior_antibiotics_30d__comp0           3658
              hx_charlson_hx_diabetes_uncomp__comp0            421
                hx_charlson_hx_renal_mod_sev__comp0            120
                    hx_charlson_hx_pulmonary__comp0             79
                     hx_charlson_hx_dementia__comp0             72
                          hx_charlson_hx_chf__comp0             55
                       hx_charlson_hx_stroke__comp0           

In [9]:
check_value_counts(
    df_events, "event_name", name="Events by phenotype component", n=20,
)

check_value_counts(
    df_events, "domain", name="Events by domain", n=20,
)


Value check: Events by phenotype component


,count
event_name,
gavaghan_hx_prior_antibiotics_fq_ceph_90d__comp1,20102
jones_hx_prior_antibiotics_30d__comp1,20102
kim_hx_recent_antibiotics_90d__comp1,20102
tumbarello_hx_recent_abx_90d__comp0,20102
kim_hx_prior_antibiotics_90d__comp0,20102
charlson_hx_diabetes_uncomp__comp2,19884
jones_hx_prior_antibiotics_30d__comp0,10098
kim_hx_recent_antibiotics_90d__comp0,10098
charlson_hx_diabetes_uncomp__comp0,828



Value check: Events by domain


,count
domain,
prescribing,140590
problems,2217
diagnosis,3
